# SAA+ Benchmark on MVTec and VisA

Reproduces image-AUROC / pixel-AUROC / AP / F1 from the SAA+ paper  
(~500 images per dataset, stratified sampling, T4 GPU)

In [ ]:
%cd /content
!git clone -b SAA-plus https://github.com/SyDuc7421/Segment-Any-Anomaly.git
%cd Segment-Any-Anomaly/

import re, pathlib
for p in pathlib.Path('GroundingDINO').rglob('*.py'):
    txt = p.read_text()
    patched = re.sub(r'transformers[^"\']*<4\.\d+\.\d+', 'transformers>=4.41.0', txt)
    if patched != txt:
        p.write_text(patched)

%cd GroundingDINO/
!pip install -q -e . --no-build-isolation
%cd ../SAM
!pip install -q -e .
!pip install -q "transformers>=4.41.0" "supervision>=0.6.0,<0.21.0" \
    opencv-python pycocotools matplotlib onnxruntime onnx ipykernel gradio loguru
%cd ..

In [ ]:
# Restart so updated transformers is loaded from disk
import os
os.kill(os.getpid(), 9)

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p weights
%cd weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
%cd ..

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p /content/datasets

# MVTec AD — official mirror (~4.9 GB)
!wget -q -O /content/datasets/mvtec.tar.xz \
    "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz"
!tar -xf /content/datasets/mvtec.tar.xz -C /content/datasets/

import os
os.environ['MVTEC_DIR'] = '/content/datasets/mvtec_anomaly_detection'
print('MVTec classes:', sorted(os.listdir(os.environ['MVTEC_DIR'])))

In [ ]:
# VisA — download via Kaggle API (requires kaggle.json uploaded to /root/.kaggle/)
# Upload kaggle.json first: Files panel → upload → /root/.kaggle/kaggle.json
!pip install -q kaggle
!mkdir -p ~/.kaggle && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d anomalib/visa -p /content/datasets/ --unzip

import os
os.environ['VISA_DIR'] = '/content/datasets/VisA_pytorch/1cls'
print('VisA classes:', sorted(os.listdir(os.environ['VISA_DIR'])))

In [ ]:
%cd /content/Segment-Any-Anomaly
import os, subprocess

os.environ['MVTEC_DIR'] = '/content/datasets/mvtec_anomaly_detection'

# MAX_SAMPLES=34 → ~510 images across 15 classes
result = subprocess.run(
    ['python', 'run_MVTec.py'],
    env={**os.environ, 'MAX_SAMPLES': '34'}
)
print('MVTec benchmark done, exit code:', result.returncode)

In [ ]:
%cd /content/Segment-Any-Anomaly
import os, subprocess

os.environ['VISA_DIR'] = '/content/datasets/VisA_pytorch/1cls'

# MAX_SAMPLES=42 → ~504 images across 12 classes
result = subprocess.run(
    ['python', 'run_VisA_public.py'],
    env={**os.environ, 'MAX_SAMPLES': '42'}
)
print('VisA benchmark done, exit code:', result.returncode)

In [ ]:
%cd /content/Segment-Any-Anomaly
import pandas as pd, glob

def summarize(csv_path, label):
    df = pd.read_csv(csv_path, index_col=0)
    mean_row = df.mean(numeric_only=True).rename('MEAN')
    df = pd.concat([df, mean_row.to_frame().T])
    cols = [c for c in ['i_roc','p_roc','i_ap','p_ap','i_f1','p_f1'] if c in df.columns]
    print(f'\n{"="*60}\n  {label}\n{"="*60}')
    print(df[cols].to_string(float_format='{:.2f}'.format))

for p in glob.glob('result/csv/mvtec-indx-*.csv'):
    summarize(p, f'MVTec — {p}')
for p in glob.glob('result/csv/visa_public-indx-*.csv'):
    summarize(p, f'VisA — {p}')